# Convert Ionospheric Delay Data to CSV Format

Recommended workflow
1. Run `generate_labels.py` to generate csv and pickle files required for raytracing
2. Run `run_raytrace_batch.py` to compute the raytracing in c++ (for L1 signals)
3. Run `convert_L1_L5.ipynb` to convert the delays for L1 to L5 
4. Run `run_raytrace_batch.py` for L5 signals that are not covered by convergion
5. Run `convert_ionodata.ipynb` to convert the raytracing results to files to pandas dataframe <-------------- This File
6. Run `generate_timestep_df.ipynb` to summarize the measurement data per timestep
7. 

## Part 1: Convert Raytracing Result CSV to Pandas Dataframe

In [ ]:
%load_ext autoreload
%autoreload 2

import numpy as np
import os
import pylupnt as pnt
import pandas as pd
from src.postprocess_ionodata import *
import matplotlib.pyplot as plt

## Load L1 data

In [ ]:
datapath = os.path.join(pnt.get_output_dir(), "iono_delay", "labels_csv")
raytrace_dir = os.path.join(pnt.get_output_dir(), "iono_delay", "raytrace_csv")
savedir = os.path.join(pnt.get_output_dir(), "iono_delay", "raytrace_summary")

sim_params = {
    "gnss_const": "GPS",
    "signal_family": 1,
    "lcrns_idx": 0,
    "epoch_ymdh": [2025, 3, 1, 12],
    "rz12": 50.0,  # -1.0 for historical or projected R12
    "kp": 3.0,
    "n_orbit": 6,
    "dt": 1,  # time step in seconds
    "dt_raytrace": 120,  # raytracing time step in seconds
}

In [ ]:
# load L1 data
df_L1_sat0 = load_raytrace_csv_data(
    sim_params, datapath, raytrace_dir, savedir, overwrite=False, load_correction=True
)

In [ ]:
df_L1_sat0[:20]

In [ ]:
plot_labels_data(df_L1_sat0, ["GPS"])
plot_delay_data(df_L1_sat0, ["GPS"])

### Galileo E1

In [ ]:
sim_params["gnss_const"] = "GALILEO"
sim_params["signal_family"] = 1
sim_params["lcrns_idx"] = 0
df_E1_sat0 = load_raytrace_csv_data(
    sim_params, datapath, raytrace_dir, savedir, overwrite=False, load_correction=True
)
df_L1E1_sat0 = pd.concat([df_L1_sat0, df_E1_sat0], ignore_index=True)
plot_labels_data(df_L1E1_sat0, ["GPS", "GALILEO"])
plot_delay_data(df_L1E1_sat0, ["GPS", "GALILEO"], [1])

### QZSS L1

In [ ]:
sim_params["gnss_const"] = "QZSS"
sim_params["signal_family"] = 1
sim_params["lcrns_idx"] = 0
df_Q1_sat0 = load_raytrace_csv_data(
    sim_params, datapath, raytrace_dir, savedir, overwrite=False, load_correction=True
)

In [ ]:
df_L1E1Q1_sat0 = pd.concat([df_L1E1_sat0, df_Q1_sat0], ignore_index=True)

plot_labels_data(df_L1E1Q1_sat0, ["GPS", "GALILEO", "QZSS"], inv=10)

In [ ]:
plot_delay_data(df_L1E1Q1_sat0, ["GPS", "GALILEO", "QZSS"], [1])

### GPS L5

In [ ]:
sim_params["gnss_const"] = "GPS"
sim_params["signal_family"] = 5
sim_params["lcrns_idx"] = 0
df_L5_sat0 = load_raytrace_csv_data(
    sim_params, datapath, raytrace_dir, savedir, overwrite=False, load_correction=True
)
df_L1L5_sat0 = pd.concat([df_L1_sat0, df_L5_sat0], ignore_index=True)
plot_delay_data(df_L1L5_sat0, ["GPS"], [1, 5])

### Galileo E5a

In [ ]:
sim_params["gnss_const"] = "GALILEO"
sim_params["signal_family"] = 5
sim_params["lcrns_idx"] = 0
df_E5_sat0 = load_raytrace_csv_data(
    sim_params, datapath, raytrace_dir, savedir, overwrite=False, load_correction=True
)
df_E1E5_sat0 = pd.concat([df_E1_sat0, df_E5_sat0], ignore_index=True)
plot_delay_data(df_E1E5_sat0, ["GALILEO"], [1, 5])

### QZSS L5

In [ ]:
sim_params["gnss_const"] = "QZSS"
sim_params["signal_family"] = 5
sim_params["lcrns_idx"] = 0
df_Q5_sat0 = load_raytrace_csv_data(
    sim_params, datapath, raytrace_dir, savedir, overwrite=False, load_correction=True
)
df_Q1Q5_sat0 = pd.concat([df_Q1_sat0, df_Q5_sat0], ignore_index=True)
plot_labels_data(df_Q1Q5_sat0, ["QZSS"])
plot_delay_data(df_Q1Q5_sat0, ["QZSS"], [1, 5])